In [1]:
from pathlib import Path
import numpy as np

from concurrent.futures import ThreadPoolExecutor

Load Fish Arrays and Responders

In [3]:
#low mem method to load fish data and responders
base_dir = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
proj_id = "hcrt-trpv1_huc-h2b-g8m_csn_120min"

PHASIC_DPRIME_THRESH = 0.5


#Expt fish list
expt_fish_list = [
            "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
            "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
            "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
            "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
            "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
            "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
            "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
            "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
            "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]


def process_fish(fish_id):
    fish_dir = base_dir / proj_id / fish_id
    
    try:
        f_tonic = np.load(fish_dir / "f_tonic.npy", mmap_mode="r")
        f_phasic = np.load(fish_dir / "f_phasic.npy", mmap_mode="r")
        
      
        tonic_pos_idx = np.load(fish_dir / "tonic_pos_glm_iaaft_nullp99_idxs.npy")
        tonic_neg_idx = np.load(fish_dir / "tonic_neg_glm_iaaft_nullp99_idxs.npy")
        dprime = np.load(fish_dir / "phasic_dprime_cells_raw.npy")

      
        all_tonic_idx = np.union1d(tonic_pos_idx, tonic_neg_idx)
        all_phasic_idx = np.where(np.abs(dprime) >= PHASIC_DPRIME_THRESH)[0]

     
        tonic_slice = np.array(f_tonic[all_tonic_idx, :])
        phasic_slice = np.array(f_phasic[all_phasic_idx, :])

        return fish_id, tonic_slice, phasic_slice

    except FileNotFoundError:
        return None

# Main Execution
tonic_data = {}
phasic_data = {}

print("Started processing fish folders")


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish, expt_fish_list)

for result in results:
    if result is not None:
        fish_id, t_res, p_res = result
        tonic_data[fish_id] = t_res
        phasic_data[fish_id] = p_res

print(f"Completed processing for {len(tonic_data)} fish.")


Started processing fish folders
Completed processing for 9 fish.


In [6]:
from scipy.stats import zscore
for fish in expt_fish_list:
    fish_name = ("hcrt-trpv1_huc-h2b-g8m_csn_120min", fish)
    print(f"Processing fish: {fish_name}")

    if fish_name in phasic_data:
        #  Filtered matrix (Cells x Timepoints)
        raw_phasic_traces = phasic_data[fish_name]

        # Z-score normalization along the time axis (axis=1)
        # Set each cell's mean firing rate to 0 and standard deviation to 1
        z_phasic_traces = zscore(raw_phasic_traces, axis=1)

    
        print(f"Normalization Complete for {fish_name}")
        print(f"Z-scored Matrix Shape: {z_phasic_traces.shape}")
        print(f"Verification - Mean should be about 0 more or less: {z_phasic_traces[0].mean():.3f}")
        print(f"Verification - Standard deviation should be exactly 1: {z_phasic_traces[0].std():.3f}")

    
    else:
        print(f"fish data not found for {fish_name}")


Processing fish: ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4')
fish data not found for ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4')
Processing fish: ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1')
fish data not found for ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1')
Processing fish: ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2')
fish data not found for ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2')
Processing fish: ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1')
fish data not found for ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1')
Processing fish: ('hcrt-trpv1_huc-h2b-g8m_csn_120min', '251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2')
fish data not found for ('hcrt-trpv1_huc-h2b-

In [ ]:
from scipy.stats import zscore
from sklearn.decomposition import FactorAnalysis
import matplotlib.pyplot as plt

target_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4'
raw_phasic_traces = phasic_data[target_fish]


z_phasic_traces = zscore(raw_phasic_traces, axis=1)
print(f"Z-score Normalization complete! Shape: {z_phasic_traces.shape}")


n_factors = 20
fa = FactorAnalysis(n_components=n_factors, random_state=42)
latent_factors = fa.fit_transform(z_phasic_traces.T)

print(f"Factor Analysis complete! Latent factor matrix shape: {latent_factors.shape}")


plt.figure(figsize=(12, 5))
for i in range(3):
    plt.plot(latent_factors[:, i], label=f"Temporal Pattern (Factor) {i+1}")

plt.title(f"Top 3 Hidden Neural Activity Trends for {target_fish}", fontsize=14)
plt.xlabel("Timepoints (Volumes)", fontsize=12)
plt.ylabel("Normalized Firing Amplitude", fontsize=12)
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.show()


Z-score Normalization complete! Shape: (5861, 7200)


ValueError: Input X contains NaN.
FactorAnalysis does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [25]:
import os

ideal parallel analysis but too computationally expensive

In [43]:
import sys
import numpy as np

sys.path.append('/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/utils')

import horns

data = z_phasic_traces.T

print(f"Module found! Running Horn's Analysis on matrix shape: {data.shape}...")

optimal_k = horns.parallel_analysis(
    data, 
    simulations=100, 
    randomisation_method="bootstrap", 
    analysis_type="fa", 
    full_output=False
)

print("\n")
print(f" Optimal number of latent factors for FA: {optimal_k}")


Module found! Running Horn's Analysis on matrix shape: (7200, 5861)...


/ssd-pool/james/conda_envs/FA_analysis/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


KeyboardInterrupt: 

In [45]:
import sys
import numpy as np


sys.path.append('/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/utils')
import horns

data = z_phasic_traces.T



optimal_k = horns.parallel_analysis(
    data, 
    simulations=5,                 # Reduced to 5 for sake of processing time. This may reduce accuracy, but due to the large number of cells, it should still provide a reasonable estimate.
    randomisation_method="bootstrap", 
    analysis_type="fa",           
    full_output=False
)


print(f" Optimal number of latent factors for FA: {optimal_k}")



KeyboardInterrupt: 

In [46]:
import sys
import numpy as np


sys.path.append('/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/utils')
import horns

data = z_phasic_traces.T



optimal_k = horns.parallel_analysis(
    data, 
    simulations=5,                 # Reduced to 5 for sake of processing time. This may reduce accuracy, but due to the large number of cells, it should still provide a reasonable estimate.
    randomisation_method="bootstrap", 
    analysis_type="pca",           # Switched to 'pca' for fast matrix calculations
    full_output=False
)


print(f" Optimal number of latent factors for PCA: {optimal_k}")



KeyboardInterrupt: 

In [47]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt


data = z_phasic_traces.T

print(f"Running optimized SVD Factor evaluation on matrix shape: {data.shape}...")


n_eval_components = 50
svd = TruncatedSVD(n_components=n_eval_components, random_state=42)
svd.fit(data)

explained_variance = svd.explained_variance_ratio_

plt.figure(figsize=(10, 5))
plt.plot(range(1, n_eval_components + 1), explained_variance, 'o-', linewidth=2, color='#1f77b4')

plt.title("Scree Plot: Identifying Optimal Factor Count for Phasic Traces", fontsize=14, pad=15)
plt.xlabel("Factor Number (Component Index)", fontsize=12)
plt.ylabel("Proportion of Variance Explained", fontsize=12)
plt.grid(True, alpha=0.3)


plt.axvline(x=20, color='r', linestyle='--', label="Yun's Baseline Blueprint (K=20)")
plt.legend(loc="upper right")

plt.show()

cumulative_variance = np.cumsum(explained_variance)
print(f"Top 5 factors explain {cumulative_variance[4]*100:.1f}% of total neural variance.")
print(f"Top 20 factors explain {cumulative_variance[19]*100:.1f}% of total neural variance.")


Running optimized SVD Factor evaluation on matrix shape: (7200, 5861)...


ValueError: Input X contains NaN.
TruncatedSVD does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values